In [0]:
from pyspark.sql import functions as F

In [0]:
catalog = "proyecto"
schema = "silver"
schema_bronze = "bronze"
silver = 'abfss://silver@adlscshm.dfs.core.windows.net/instacart/'

In [0]:
# lectura de la data
df_aisles = spark.table(f"{catalog}.{schema_bronze}.aisles")
df_departments = spark.table(f"{catalog}.{schema_bronze}.departments")
df_order_products = spark.table(f"{catalog}.{schema_bronze}.order_products") 
df_orders = spark.table(f"{catalog}.{schema_bronze}.orders")
df_products = spark.table(f"{catalog}.{schema_bronze}.products")

In [0]:
df_orders_silver = (
    df_orders
    .withColumnRenamed("order_id", "cod_order")
    .withColumnRenamed("user_id", "cod_user")
    .withColumnRenamed("eval_set", "des_eval_set")
    .withColumnRenamed("order_number", "val_order_number")
    .withColumnRenamed("order_dow", "val_order_dow")
    .withColumnRenamed("order_hour_of_day", "val_order_hour_of_day")
    .withColumnRenamed("days_since_prior_order", "val_days_since_prior_order")
)

df_order_products_silver = (
    df_order_products
    .withColumnRenamed("order_id", "cod_order")
    .withColumnRenamed("product_id", "cod_product")
    .withColumnRenamed("add_to_cart_order", "val_add_to_cart_order")
    .withColumnRenamed("reordered", "val_reordered")
)

df_products_silver = (
    df_products
    .join(df_aisles, "aisle_id", "left")
    .join(df_departments, "department_id", "left")
)
# drop _ingest_ts
df_products_silver = df_products_silver.drop("_ingest_ts")

df_products_silver = (
    df_products_silver
    # IDs / códigos
    .withColumnRenamed("product_id", "cod_product")
    .withColumnRenamed("aisle_id", "cod_aisle")
    .withColumnRenamed("department_id", "cod_department")
    # Descripciones
    .withColumnRenamed("product_name", "des_product_name")
    .withColumnRenamed("aisle", "des_aisle")
    .withColumnRenamed("department", "des_department")
)

In [0]:
df_orders_silver = (
    df_orders_silver
    
    .withColumn("flg_weekend", F.when(F.col("val_order_dow").isin(0,6), F.lit(1)).otherwise(F.lit(0)))
    
    .withColumn(
        "cat_daypart",
        F.when(F.col("val_order_hour_of_day").between(0,5),  F.lit("madrugada"))
         .when(F.col("val_order_hour_of_day").between(6,11), F.lit("mañana"))
         .when(F.col("val_order_hour_of_day").between(12,17),F.lit("tarde"))
         .otherwise(F.lit("noche"))
    )
    
    .withColumn("flg_first_order", F.when(F.col("val_order_number")==1, F.lit(1)).otherwise(F.lit(0)))
    
    .withColumn("cat_recencia", F.when(F.col("val_days_since_prior_order") > 5, F.lit("alto")).otherwise(F.lit("bajo")))
)



df_order_products_silver = (
    df_order_products_silver
    .withColumn("cat_prioridad", F.when(F.col("val_add_to_cart_order") > 5, F.lit("alto")).otherwise(F.lit("bajo")))
)





In [0]:
df_orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").option("path", f"{silver}orders").saveAsTable(f"{catalog}.{schema}.orders")

df_order_products_silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").option("path", f"{silver}order_products").saveAsTable(f"{catalog}.{schema}.order_products")

df_products_silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").option("path", f"{silver}products_enriched").saveAsTable(f"{catalog}.{schema}.products_enriched")


In [0]:
%sql
select * 
from proyecto.silver.products_enriched